https://deepeval.com/guides/guides-using-custom-llms
https://deepeval.com/integrations/models/ollama


In [2]:
 !ollama list

NAME               ID              SIZE      MODIFIED      
qwen3:1.7b         8f68893c685c    1.4 GB    3 days ago       
llama3.2:1b        baf6a787fdff    1.3 GB    6 days ago       
qwen3:0.6b         7df6b6e09427    522 MB    3 months ago     
qwen3:latest       500a1f067a9f    5.2 GB    3 months ago     
qwen2.5:latest     845dbda0ea48    4.7 GB    8 months ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    11 months ago    


In [1]:
!deepeval set-ollama qwen3:1.7b

Settings updated for this session. To persist, use --save=dotenv[:path] (default
.env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen3:1.7b` for all 
evals that require an LLM.


In [4]:

# Ollama cloud API Key
path = "../../ollama-cloud-key.txt"
with open(path) as f:
    # Read the contents of the file into a variable
    api_key = f.read()

In [5]:
import os
from ollama import Client

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + api_key})

messages = [
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
]
response = ''
for part in client.chat('gpt-oss:120b', messages=messages, stream=True):
    # if part.choices[0].delta.content is not None:
    #     response += part.choices[0].delta.content
    #print('part')
    # answer += part['message']['content']
    #print(part['message']['content'], end='', flush=True)
    response += part['message']['content']
newl = response.find('\n')
print(response[:newl])

**Short answer:** The sky looks blue because the Earth’s atmosphere scatters short‑wavelength (blue and violet) sunlight much more efficiently than it scatters longer‑wavelength (red, orange, yellow) light. Our eyes are more sensitive to blue than violet, and some of the violet light is absorbed by the upper atmosphere, so the net result is a blue sky.


In [11]:
import transformers
import torch
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models import DeepEvalBaseLLM


class CustomLlama3_8B(DeepEvalBaseLLM):
    def __init__(self):
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        model_4bit = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Meta-Llama-3-8B-Instruct",
            device_map="auto",
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(
            "meta-llama/Meta-Llama-3-8B-Instruct"
        )

        self.model = model_4bit
        self.tokenizer = tokenizer

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        model = self.load_model()

        pipeline = transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=self.tokenizer,
            use_cache=True,
            device_map="auto",
            max_length=2500,
            do_sample=True,
            top_k=5,
            num_return_sequences=1,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        return pipeline(prompt)

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Llama-3 8B"


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/piek/.pyenv/versions/3.10.11/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/piek/.pyenv/versions/3.10.11/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/piek/.pyenv/versions/3.10.11/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/piek/.pyenv/versions/3.10.11/lib/python3.10/site-packages/traitlets/config/application.py", lin

AttributeError: _ARRAY_API not found

RuntimeError: Failed to import transformers.models.auto.modeling_auto because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
numpy.core.multiarray failed to import

In [6]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase
from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval import evaluate

In [9]:
answer_relevancy = AnswerRelevancyMetric(model=client, key=api_key)

test_case = LLMTestCase(input="What do you like", actual_output="I love cats")
evaluate(test_cases=[test_case], metrics=[answer_relevancy])

TypeError: AnswerRelevancyMetric.__init__() got an unexpected keyword argument 'key'

In [10]:
#!deepeval set-ollama qwen3-coder:480b-cloud
!deepeval set-ollama qwen3:1.7b


model_name = "qwen3:1.7b"
#model_name = "deepseek-v3.1:671b-cloud"
#model_name="qwen3-coder:480b-cloud"
model = OllamaModel(
    # host="https://ollama.com",
    # headers={'Authorization': 'Bearer ' + key}
    model=model_name,
    base_url="http://localhost:11434",
    temperature=0.1
)

answer_relevancy = AnswerRelevancyMetric(model=model)

test_case = LLMTestCase(input="What do you like", actual_output="I love cats")
evaluate(test_cases=[test_case], metrics=[answer_relevancy])

Settings updated for this session. To persist, use --save=dotenv[:path] (default
.env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen3:1.7b` for all 
evals that require an LLM.


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 0.6666666666666666, threshold: 0.5, strict: False, evaluation model: qwen3:1.7b (Ollama), reason: The score is 0.67 because there is one irrelevant statement, 'The statement 'I' is too vague and does not provide specific information about what the user likes,' which is minor and does not significantly affect relevance. The output remains at this score as it avoids major irrelevance while addressing the input's question., error: None)

For test case:

  - input: What do you like
  - actual output: I love cats
  - expected output: None
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=115594;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 295.16s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.6666666666666666, reason="The score is 0.67 because there is one irrelevant statement, 'The statement 'I' is too vague and does not provide specific information about what the user likes,' which is minor and does not significantly affect relevance. The output remains at this score as it avoids major irrelevance while addressing the input's question.", strict_mode=False, evaluation_model='qwen3:1.7b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "I",\n    "love",\n    "cats"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "The statement \'I\' is too vague and does not provide specific information about what the user likes."\n    },\n    {\n        "verdict": "idk",\n        "reason": "The statement \'love\' is ambiguous as it does not specify what is being loved, making i

In [30]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase

criteria = """Coherence (1-5) - the collective quality of all sentences. We align this dimension with
the DUC quality question of structure and coherence whereby the summary should be
well-structured and well-organized. The summary should not just be a heap of related information, but should build from sentence to sentence to a coherent body of information about a topic."""

coherence_metric = GEval(
    name="Coherence",
    model=model,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
)

# Now define your test case, actual_output is your LLM output
test_case = LLMTestCase(input="Hey how's the weather like today?", actual_output="It's alright!", expected_output="Rainy")

# Use G-Eval metric
coherence_metric.measure(test_case)
print(coherence_metric.score, coherence_metric.reason)

Output()

0.0 The actual output 'It's alright!' does not match the expected 'Rainy' in content and structure. The sentence flow and logical progression are inconsistent, failing to align with the expected theme of weather description. The coherence and thematic unity are completely disrupted.


In [24]:
#https://medium.com/@jeffreyip54/you-can-now-use-ollama-for-llm-as-a-judge-76f06e3005c9

#from deepeval.dataset import EvaluationDataset

from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
#  AnswerRelevnacyMetric,
  FaithfulnessMetric
)

contextual_precision = ContextualPrecisionMetric(model=model)
contextual_recall = ContextualRecallMetric(model=model)
contextual_relevancy = ContextualRelevancyMetric(model=model)
answer_relevancy = AnswerRelevancyMetric(threshold=0.8, model=model)
faithfulness = FaithfulnessMetric(model=model)

#dataset = EvaluationDataset()

evaluate(test_cases=[test_case], metrics=[contextual_precision, contextual_recall, contextual_relevancy, answer_relevancy, faithfulness])

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

Output()

MissingTestCaseParamsError: 'retrieval_context' and 'expected_output' cannot be None for the 'Contextual Precision' metric

In [19]:
host="https://ollama.com",
headers={'Authorization': 'Bearer ' + key}
model = 'gpt-oss:120b'
!deepeval set-local-model=model  --base-url=host --api-key=key

Usage: deepeval [OPTIONS] COMMAND [ARGS]...
Try 'deepeval --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'set-local-model=model'.                                     │
╰──────────────────────────────────────────────────────────────────────────────╯
